# Weather Hazard Category (WHC): preprocessing

This notebook creates the daily Weather Hazard Category target for UTS 36120 Advanced ML Assessment Task 2. It reads the immutable hourly Sydney archive data, aggregates it to daily observations, calculates the Weather Hazard Index (WHI), maps it to the required four classes, then aligns the class target to an **exact +7-day** forecast horizon.

No future observation is used as a feature. This notebook only produces dated targets for later feature engineering and classification modelling.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

pd.set_option("display.float_format", "{:.2f}".format)


def project_root(start: Path | None = None) -> Path:
    """Return the project directory containing ``pyproject.toml``."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not locate project root from the current directory.")


PROJECT_ROOT = project_root()
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "sydney_hourly.csv"
INTERIM_PATH = PROJECT_ROOT / "data" / "interim" / "weather_hazard" / "sydney_daily_whc_inputs.csv"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "weather_hazard" / "sydney_whc_targets.csv"

WHC_COLUMNS = ["precipitation", "wind_gusts_10m", "cloud_cover", "snowfall", "temperature_2m"]
WHC_DAILY_AGGREGATIONS = {
    "precipitation": "sum",
    "wind_gusts_10m": "max",
    "cloud_cover": "mean",
    "snowfall": "sum",
    "temperature_2m": "mean",
}
FORECAST_HORIZON_DAYS = 7

## Daily aggregation

The brief specifies daily total precipitation and snowfall, daily maximum wind gusts, and daily mean cloud cover and temperature. The Weather Hazard Index must be calculated from these daily aggregates, not by averaging hourly hazard scores.

In [2]:
hourly = pd.read_csv(RAW_PATH, parse_dates=["time"])
missing_columns = sorted(set(["time", *WHC_COLUMNS]) - set(hourly.columns))
if missing_columns:
    raise ValueError(f"Source is missing required WHC columns: {missing_columns}")
if hourly[WHC_COLUMNS].isna().any().any():
    raise ValueError("Source contains missing WHC inputs; resolve before calculating targets.")
if hourly["time"].duplicated().any():
    raise ValueError("Source contains duplicate timestamps; resolve before daily aggregation.")

daily_whc_inputs = (
    hourly.set_index("time")[WHC_COLUMNS]
    .sort_index()
    .resample("D")
    .agg(WHC_DAILY_AGGREGATIONS)
)
if daily_whc_inputs.isna().any().any():
    raise ValueError("Daily WHC inputs contain missing values after aggregation.")

INTERIM_PATH.parent.mkdir(parents=True, exist_ok=True)
daily_whc_inputs.to_csv(INTERIM_PATH, index_label="date")
print(f"Daily input window: {daily_whc_inputs.index.min():%Y-%m-%d} to {daily_whc_inputs.index.max():%Y-%m-%d}")
print(f"Daily rows: {len(daily_whc_inputs):,}")
daily_whc_inputs.head()

Daily input window: 2016-01-01 to 2025-12-31
Daily rows: 3,653


,precipitation,wind_gusts_10m,cloud_cover,snowfall,temperature_2m
time,,,,,
2016-01-01,0.00,34.90,64.12,0.00,20.23
2016-01-02,0.00,35.30,99.29,0.00,20.24
2016-01-03,1.30,40.70,99.96,0.00,20.67
2016-01-04,15.60,34.20,99.04,0.00,19.22
2016-01-05,52.30,40.70,99.83,0.00,18.60


## WHI calculation and class mapping

The five hazard scores, weights, and four WHC class boundaries below are transcribed from the assessment brief. `whi` is retained for auditing only; the machine-learning target is the categorical `whc` class.

In [3]:
def calculate_whc(daily: pd.DataFrame) -> pd.DataFrame:
    """Add WHI components, continuous index, and required integer WHC class."""
    result = daily.copy()
    result["rain_hazard"] = (result["precipitation"] / 30).clip(upper=1)
    result["wind_hazard"] = (result["wind_gusts_10m"] / 100).clip(upper=1)
    result["cloud_hazard"] = (result["cloud_cover"] / 100).clip(upper=1)
    result["snow_hazard"] = (result["snowfall"] / 15).clip(upper=1)
    result["temp_hazard"] = ((result["temperature_2m"] - 22).abs() / 25).clip(upper=1)
    result["whi"] = 100 * (
        0.30 * result["rain_hazard"]
        + 0.30 * result["wind_hazard"]
        + 0.20 * result["cloud_hazard"]
        + 0.10 * result["snow_hazard"]
        + 0.10 * result["temp_hazard"]
    )
    result["whc"] = pd.cut(
        result["whi"],
        bins=[float("-inf"), 25, 50, 75, float("inf")],
        labels=[0, 1, 2, 3],
        right=False,
    ).astype("int8")
    return result


daily_whc = calculate_whc(daily_whc_inputs)
daily_whc[["whi", "whc", "rain_hazard", "wind_hazard", "cloud_hazard", "snow_hazard", "temp_hazard"]].describe()

,whi,whc,rain_hazard,wind_hazard,cloud_hazard,snow_hazard,temp_hazard
count,3653.00,3653.00,3653.00,3653.00,3653.00,3653.00,3653.00
mean,27.14,0.56,0.08,0.43,0.49,0.00,0.20
std,10.82,0.59,0.19,0.13,0.29,0.00,0.14
min,8.06,0.00,0.00,0.13,0.00,0.00,0.00
25%,19.66,0.00,0.00,0.34,0.25,0.00,0.07
50%,25.35,1.00,0.01,0.41,0.48,0.00,0.18
75%,31.76,1.00,0.07,0.50,0.72,0.00,0.33
max,83.10,3.00,1.00,1.00,1.00,0.00,0.57


## Forecast-target alignment and class balance

For a feature row dated *t*, `whc_target` is the category observed on *t* + 7 calendar days. The final seven source dates have no future target and are removed. Class frequencies are inspected before modelling because sparse classes require time-aware evaluation and may constrain model selection.

In [4]:
whc_targets = pd.DataFrame(
    {
        "feature_date": daily_whc.index,
        "target_date": daily_whc.index + pd.Timedelta(days=FORECAST_HORIZON_DAYS),
        "whc_target": daily_whc["whc"].shift(-FORECAST_HORIZON_DAYS).to_numpy(),
        "whi_target": daily_whc["whi"].shift(-FORECAST_HORIZON_DAYS).to_numpy(),
    }
).dropna(subset=["whc_target"])
whc_targets["whc_target"] = whc_targets["whc_target"].astype("int8")

if not (whc_targets["target_date"] > whc_targets["feature_date"]).all():
    raise AssertionError("WHC target dates must be later than feature dates.")
if not whc_targets["whc_target"].isin([0, 1, 2, 3]).all():
    raise AssertionError("WHC targets must be one of the four required classes.")

PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
whc_targets.to_csv(PROCESSED_PATH, index=False)

class_labels = {0: "Low Risk", 1: "Moderate Risk", 2: "High Risk", 3: "Extreme Risk"}
class_balance = (
    whc_targets["whc_target"]
    .value_counts(sort=False)
    .reindex(class_labels)
    .rename_axis("class")
    .to_frame("rows")
    .assign(label=lambda frame: frame.index.map(class_labels), pct=lambda frame: frame["rows"] / len(whc_targets) * 100)
)

print(f"Saved daily WHC inputs: {INTERIM_PATH.relative_to(PROJECT_ROOT)}")
print(f"Saved +{FORECAST_HORIZON_DAYS}-day WHC targets: {PROCESSED_PATH.relative_to(PROJECT_ROOT)}")
print(f"Model-ready target rows: {len(whc_targets):,}")
display(class_balance)
whc_targets.head()

Saved daily WHC inputs: data/interim/sydney_daily_whc_inputs.csv
Saved +7-day WHC targets: data/processed/sydney_whc_targets.csv
Model-ready target rows: 3,646


,rows,label,pct
class,,,
0,1777,Low Risk,48.74
1,1711,Moderate Risk,46.93
2,150,High Risk,4.11
3,8,Extreme Risk,0.22


,feature_date,target_date,whc_target,whi_target
0,2016-01-01,2016-01-08,0,20.49
1,2016-01-02,2016-01-09,0,17.98
2,2016-01-03,2016-01-10,0,13.93
3,2016-01-04,2016-01-11,0,12.88
4,2016-01-05,2016-01-12,1,29.79


## Output contract

- `data/interim/sydney_daily_whc_inputs.csv`: daily inputs and daily aggregation results for auditability.
- `data/processed/sydney_whc_targets.csv`: `feature_date`, `target_date`, `whc_target` (the required class target), and `whi_target` (audit value only).

Rows retain chronological order. Build predictors exclusively from information available on or before `feature_date`, then use time-based training, validation, and test splits without shuffling.